# Detecting Topic Merges and Splits in Dynamic Political Conversations

Cláudia Oliveira 

Supervisor - Prof. Dr. Álvaro Figueira

Faculty of Science, University of Porto

Saving dataset

In [ ]:
import os
import pandas as pd
import os
os.chdir("../..")

folder_path = "datasets/stateofunion"

data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        year = filename.split("_")[-1].replace(".txt", "")
        
        file_path = os.path.join(folder_path, filename)
        
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        
        data.append({
            "year": int(year),
            "text": text
        })

df = pd.DataFrame(data)

print(df.head())

   year                                               text
0  1797  Gentlemen of the Senate and Gentlemen of the H...
1  1798  Gentlemen of the Senate and Gentlemen of the H...
2  1799  Gentlemen of the Senate and Gentlemen of the H...
3  1800  Gentlemen of the Senate and Gentlemen of the H...
4  1825  Fellow Citizens of the Senate and of the House...


In [ ]:
df.to_csv("datasets/stateofunion_dataset.csv", index=False)

Preprocess

In [22]:
df = pd.read_csv("./datasets/stateofunion_dataset.csv")

df = df.dropna(subset=["text"])

In [23]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

df["text"] = df["text"].fillna("").astype(str)

texts = df["text"].tolist()
vectorizer = CountVectorizer(max_df=0.7)
X = vectorizer.fit_transform(texts)

filtered_vocab = vectorizer.get_feature_names_out()
vocab_set = set(filtered_vocab)

print(f"Number of words after filtering: {len(filtered_vocab)}")

def clean_text_keep_paragraphs(text):
    paragraphs = text.split("\n\n")  # split by paragraphs

    cleaned_paragraphs = []
    for p in paragraphs:
        words = re.findall(r"\b\w+\b", p.lower())
        filtered_words = [w for w in words if w in vocab_set]

        # keep only non-empty paragraphs
        if filtered_words:
            cleaned_paragraphs.append(" ".join(filtered_words))

    # join paragraphs again using paragraph separators
    return "\n\n".join(cleaned_paragraphs)

df["clean_text"] = df["text"].apply(clean_text_keep_paragraphs)

print(df[["year", "clean_text"]].head())

Number of words after filtering: 24570
   year                                         clean_text
0  1797  gentlemen gentlemen\n\napprehensive account co...
1  1798  gentlemen gentlemen\n\nreverence resignation c...
2  1799  gentlemen gentlemen\n\npeculiar satisfaction 6...
3  1800  gentlemen gentlemen\n\nimmediately adjournment...
4  1825  taking survey concerns beloved reference subje...


In [24]:
rows = []

for _, row in df.iterrows():
    year = row["year"]
    text = row["clean_text"]

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    for p in paragraphs:
        rows.append({
            "year": year,
            "paragraph": p
        })

df_paragraphs = pd.DataFrame(rows)

print(df_paragraphs.head())
print(f"Total de parágrafos: {len(df_paragraphs)}")

   year                                          paragraph
0  1797                                gentlemen gentlemen
1  1797  apprehensive account contagious sickness affli...
2  1797  although congratulate reestablishment restorat...
3  1797  envoys extraordinary french republic embarked ...
4  1797  confidently asserted nothing occurred adjournm...
Total de parágrafos: 20918


In [25]:
import re
import spacy
import pandas as pd
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Process with spaCy
    doc = nlp(text)

    tokens = []
    for token in doc:
        # Remove stopwords, spaces, and very short tokens
        if not token.is_stop and not token.is_space:
            lemma = token.lemma_.strip()
            if lemma:
                tokens.append(lemma)

    return " ".join(tokens)

# Apply cleaning
df_paragraphs["clean_paragraph"] = df_paragraphs["paragraph"].apply(clean_text)

# 3. Remove rows with 3 words or less
df_paragraphs = df_paragraphs[
    df_paragraphs["clean_paragraph"].apply(lambda x: len(x.split()) > 3)
]

df_paragraphs = df_paragraphs.reset_index(drop=True)

print(df_paragraphs.head())
print(f"Final number of paragraphs: {len(df_paragraphs)}")

   year                                          paragraph  \
0  1797  apprehensive account contagious sickness affli...   
1  1797  although congratulate reestablishment restorat...   
2  1797  envoys extraordinary french republic embarked ...   
3  1797  confidently asserted nothing occurred adjournm...   
4  1797  indeed whatever issue negotiation france hold ...   

                                     clean_paragraph  
0  apprehensive account contagious sickness affli...  
1  congratulate reestablishment restoration perso...  
2  envoy extraordinary french republic embark jul...  
3  confidently assert occur adjournment render in...  
4  issue negotiation france hold tranquillity obt...  
Final number of paragraphs: 19283


In [26]:
# Ensure the correct columns
df_final = df_paragraphs[["year", "clean_paragraph"]].copy()

# Tokenization (split by words)
df_final["Tokens"] = df_final["clean_paragraph"].apply(lambda x: x.split())

# Rename columns
df_final = df_final.rename(columns={
    "year": "Year"
})

# Keep only the desired columns
df_final = df_final[["Year", "Tokens"]]

print(df_final.head())

   Year                                             Tokens
0  1797  [apprehensive, account, contagious, sickness, ...
1  1797  [congratulate, reestablishment, restoration, p...
2  1797  [envoy, extraordinary, french, republic, embar...
3  1797  [confidently, assert, occur, adjournment, rend...
4  1797  [issue, negotiation, france, hold, tranquillit...


In [20]:
df_final.to_csv("datasets/stateofunion_tokens.csv", index=False)